## Built-in callbacks

1. TensorBoard
2. Model checkpoints
3. Early Stopping
4. CSV Logger
5. Learning Rate Scheduler
6. ReduceLROnPlateau

In [1]:
import tensorflow as tf
import numpy as np
import tensorflow_datasets as tfds
import matplotlib.pyplot as plt
import io
from PIL import Image

from tensorflow.keras.callbacks import TensorBoard, ModelCheckpoint, EarlyStopping, CSVLogger, LearningRateScheduler, ReduceLROnPlateau
%load_ext tensorboard

import os
import math
import datetime
import pandas as pd

### Prepare the Horses vs Humans dataset

In [2]:
path = "./data"
splits, info = tfds.load('horses_or_humans',
                         data_dir=path,
                         as_supervised=True,
                         with_info=True, split=['train[:80%]', 'train[80%:]', 'test'])

(train_examples, validation_examples, test_examples) = splits
num_examples = info.splits['train'].num_examples
num_classes = info.features['label'].num_classes

2026-02-17 15:16:01.038524: I metal_plugin/src/device/metal_device.cc:1154] Metal device set to: Apple M1 Pro
2026-02-17 15:16:01.038695: I metal_plugin/src/device/metal_device.cc:296] systemMemory: 16.00 GB
2026-02-17 15:16:01.038701: I metal_plugin/src/device/metal_device.cc:313] maxCacheSize: 5.92 GB
I0000 00:00:1771321561.039011 1034382 pluggable_device_factory.cc:305] Could not identify NUMA node of platform GPU ID 0, defaulting to 0. Your kernel may not have been built with NUMA support.
I0000 00:00:1771321561.039283 1034382 pluggable_device_factory.cc:271] Created TensorFlow device (/job:localhost/replica:0/task:0/device:GPU:0 with 0 MB memory) -> physical PluggableDevice (device: 0, name: METAL, pci bus id: <undefined>)


In [3]:
SIZE = 150
IMAGE_SIZE = (SIZE, SIZE)

In [4]:
# Format the images to feed into model

def format_image(image, label):
    image = tf.image.resize(image, IMAGE_SIZE) / 255.0
    return image, label

BATCH_SIZE = 32

In [5]:
train_batches = train_examples.shuffle(num_examples//4).map(format_image).batch(BATCH_SIZE).prefetch(1)
validation_batches = validation_examples.map(format_image).batch(BATCH_SIZE).prefetch(1)
test_batches = test_examples.map(format_image).batch(1)

In [6]:
for image_batch, label_batch in train_batches.take(1):
    pass

print("Image batch shape: ",image_batch.shape)
print("Label batch shape: ",label_batch.shape)

2026-02-17 15:16:01.334326: I tensorflow/core/kernels/data/tf_record_dataset_op.cc:387] The default buffer size is 262144, which is overridden by the user specified `buffer_size` of 8388608


Image batch shape:  (32, 150, 150, 3)
Label batch shape:  (32,)


2026-02-17 15:16:01.546257: W tensorflow/core/kernels/data/cache_dataset_ops.cc:916] The calling iterator did not fully read the dataset being cached. In order to avoid unexpected truncation of the dataset, the partially cached contents of the dataset  will be discarded. This can happen if you have an input pipeline similar to `dataset.cache().take(k).repeat()`. You should use `dataset.take(k).cache().repeat()` instead.
2026-02-17 15:16:01.550851: I tensorflow/core/framework/local_rendezvous.cc:407] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


### Build model

In [7]:
def build_model(dense_units, input_shape=IMAGE_SIZE+(3,)):
    model = tf.keras.models.Sequential([
        tf.keras.layers.Conv2D(16, (3,3), activation='relu', input_shape=input_shape),
        tf.keras.layers.MaxPooling2D(2,2),
        tf.keras.layers.Conv2D(32, (3,3), activation='relu'),
        tf.keras.layers.MaxPooling2D(2,2),
        tf.keras.layers.Conv2D(64, (3,3), activation='relu'),
        tf.keras.layers.MaxPooling2D(2,2),
        tf.keras.layers.Flatten(),
        tf.keras.layers.Dense(dense_units, activation='relu'),
        tf.keras.layers.Dense(2, activation='softmax')
    ])
    return model

## TensorBoard

In [8]:
!rm -rf logs

In [14]:
model = build_model(dense_units=256)
model.compile(optimizer='sgd',
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])

logdir = os.path.join("logs", datetime.datetime.now().strftime("%Y%m%d-%H%M%S"))
tensorboard_callback = tf.keras.callbacks.TensorBoard(logdir)

model.fit(train_batches, epochs=10, validation_data=validation_batches, callbacks=[tensorboard_callback])

Epoch 1/10


/Users/opmule/miniforge3/envs/ai_tensorflow/lib/python3.10/site-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


26/26 ━━━━━━━━━━━━━━━━━━━━ 2s 55ms/step - accuracy: 0.5132 - loss: 0.6924 - val_accuracy: 0.4390 - val_loss: 0.6927
Epoch 2/10
26/26 ━━━━━━━━━━━━━━━━━━━━ 1s 37ms/step - accuracy: 0.6209 - loss: 0.6356 - val_accuracy: 0.7171 - val_loss: 0.6091
Epoch 3/10
26/26 ━━━━━━━━━━━━━━━━━━━━ 1s 37ms/step - accuracy: 0.6862 - loss: 0.5924 - val_accuracy: 0.7073 - val_loss: 0.5706
Epoch 4/10
26/26 ━━━━━━━━━━━━━━━━━━━━ 1s 37ms/step - accuracy: 0.7385 - loss: 0.5488 - val_accuracy: 0.8098 - val_loss: 0.4589
Epoch 5/10
26/26 ━━━━━━━━━━━━━━━━━━━━ 1s 37ms/step - accuracy: 0.7845 - loss: 0.4662 - val_accuracy: 0.8341 - val_loss: 0.4364
Epoch 6/10
26/26 ━━━━━━━━━━━━━━━━━━━━ 1s 37ms/step - accuracy: 0.8117 - loss: 0.4336 - val_accuracy: 0.8488 - val_loss: 0.3714
Epoch 7/10
26/26 ━━━━━━━━━━━━━━━━━━━━ 1s 36ms/step - accuracy: 0.8497 - loss: 0.3589 - val_accuracy: 0.8780 - val_loss: 0.3084
Epoch 8/10
26/26 ━━━━━━━━━━━━━━━━━━━━ 1s 37ms/step - accuracy: 0.8552 - loss: 0.3333 - val_accuracy: 0.8878 - val_loss: 0.

In [16]:
%tensorboard --logdir logs

Reusing TensorBoard on port 6006 (pid 45920), started 6:10:00 ago. (Use '!kill 45920' to kill it.)

## Model Checkpoint

### Save the model weights

In [20]:
model = build_model(dense_units=256)
model.compile(optimizer='sgd',
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])

# Save the checkpoints at the specified path
model.fit(train_batches,
          epochs=5,
          validation_data=validation_batches,
          verbose=2,
          callbacks=[tf.keras.callbacks.ModelCheckpoint('weights/weights.{epoch:02d}-{val_loss:.2f}.h5', verbose=1)])

Epoch 1/5

Epoch 1: saving model to weights/weights.01-0.70.h5


26/26 - 2s - 69ms/step - accuracy: 0.5596 - loss: 0.6813 - val_accuracy: 0.4439 - val_loss: 0.6969
Epoch 2/5

Epoch 2: saving model to weights/weights.02-0.60.h5


26/26 - 1s - 33ms/step - accuracy: 0.6484 - loss: 0.6277 - val_accuracy: 0.6049 - val_loss: 0.5974
Epoch 3/5

Epoch 3: saving model to weights/weights.03-0.70.h5


26/26 - 1s - 36ms/step - accuracy: 0.7725 - loss: 0.5583 - val_accuracy: 0.4829 - val_loss: 0.6996
Epoch 4/5

Epoch 4: saving model to weights/weights.04-0.53.h5


26/26 - 1s - 33ms/step - accuracy: 0.7786 - loss: 0.5106 - val_accuracy: 0.6390 - val_loss: 0.5320
Epoch 5/5

Epoch 5: saving model to weights/weights.05-0.37.h5


26/26 - 1s - 33ms/step - accuracy: 0.8467 - loss: 0.4130 - val_accuracy: 0.8293 - val_loss: 0.3681


### Save the model in .keras format

In [23]:
model.fit(train_batches,
          epochs=1,
          validation_data=validation_batches,
          verbose=2,
          callbacks=[ModelCheckpoint('models/saved_model.keras', verbose=1)
          ])


Epoch 1: saving model to models/saved_model.keras
26/26 - 1s - 45ms/step - accuracy: 0.9173 - loss: 0.2658 - val_accuracy: 0.9415 - val_loss: 0.2213


### Save the model in .h5 format

In [24]:
model.fit(train_batches,
          epochs=1,
          validation_data=validation_batches,
          verbose=2,
          callbacks=[ModelCheckpoint('models/saved_model.h5', verbose=1)
          ])


Epoch 1: saving model to models/saved_model.h5


26/26 - 1s - 36ms/step - accuracy: 0.9586 - loss: 0.1816 - val_accuracy: 0.9659 - val_loss: 0.1603


## EarlyStopping callback

In [25]:
model = build_model(dense_units=256)
model.compile(
    optimizer='sgd',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy'])

model.fit(train_batches,
          epochs=50,
          validation_data=validation_batches,
          verbose=2,
          callbacks=[EarlyStopping(
              patience = 3,
              min_delta = 0.05,
              baseline = 0.8,
              mode='min',
              monitor='val_loss',
              restore_best_weights = True,
              verbose = 1
          )])

Epoch 1/50


/Users/opmule/miniforge3/envs/ai_tensorflow/lib/python3.10/site-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


26/26 - 2s - 81ms/step - accuracy: 0.5462 - loss: 0.6920 - val_accuracy: 0.6098 - val_loss: 0.6337
Epoch 2/50
26/26 - 1s - 33ms/step - accuracy: 0.6727 - loss: 0.6161 - val_accuracy: 0.7854 - val_loss: 0.5797
Epoch 3/50
26/26 - 1s - 36ms/step - accuracy: 0.6800 - loss: 0.5953 - val_accuracy: 0.6195 - val_loss: 0.5931
Epoch 4/50
26/26 - 1s - 33ms/step - accuracy: 0.7506 - loss: 0.5280 - val_accuracy: 0.6439 - val_loss: 0.5572
Epoch 5/50
26/26 - 1s - 32ms/step - accuracy: 0.8029 - loss: 0.4553 - val_accuracy: 0.9024 - val_loss: 0.3871
Epoch 6/50
26/26 - 1s - 33ms/step - accuracy: 0.8394 - loss: 0.4001 - val_accuracy: 0.8634 - val_loss: 0.3351
Epoch 7/50
26/26 - 1s - 33ms/step - accuracy: 0.8650 - loss: 0.3382 - val_accuracy: 0.9610 - val_loss: 0.2402
Epoch 8/50
26/26 - 1s - 33ms/step - accuracy: 0.9392 - loss: 0.2332 - val_accuracy: 0.9610 - val_loss: 0.1954
Epoch 9/50
26/26 - 1s - 35ms/step - accuracy: 0.9221 - loss: 0.2341 - val_accuracy: 0.9610 - val_loss: 0.1719
Epoch 10/50
26/26 - 1

### Explanation

1. The Core Logic <br>
    I. *`monitor='val_loss'`*: The callback is watching the Validation Loss. This is the error rate on data the model hasn't "seen" during training.<br>

    II. *`mode='min'`*: Since you are monitoring "loss," lower is better. The callback will stop training when the loss stops decreasing. (If you were monitoring val_accuracy, you would use mode='max').

<br>

2. The Thresholds <br>
    I. *`baseline = 0.8`*: This is the "starting hurdle." The model must reach a validation loss of 0.8 or lower before the patience timer even starts. If the model    never gets below 0.8, it will keep training (unless it hits the max epochs). <br>

    II. *`min_delta = 0.05`*: This defines what counts as a "real" improvement. If the loss decreases by only 0.01, this callback considers that "no improvement" because it's less than 0.05. It prevents the model from crawling forward with tiny, meaningless gains.

<br>

3. The "Patience" and Cleanup <br>
    I. *`patience = 3`*: This is the "three strikes" rule. If the model goes 3 consecutive epochs without an improvement of at least min_delta, the training stops.<br>

    II. *`restore_best_weights = True`*: This is crucial. Usually, when the model stops, it's actually worse than it was 3 epochs ago (because it spent 3 epochs failing to improve). This setting automatically rolls the model back to the version that had the absolute lowest val_loss. <br>

    III. *`verbose = 1`*: This ensures that when the training stops early, Keras prints a message in your console explaining exactly why and at which epoch it happened.

## LearningRateScheduler

At the beginning of every epoch, this callback gets the updated learning rate value from schedule function provided at __init__, with the current epoch and current learning rate, and applies the updated learning rate on the optimizer.



In [26]:
model = build_model(dense_units=256)
model.compile(
    optimizer='sgd',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy'])

In [28]:
def step_decay_with_epochs_drop(epoch):
    initial_lr = 0.01
    drop = 0.5
    epochs_drop = 1
    lr = initial_lr * math.pow(drop, math.floor((1+epoch)/epochs_drop))
    return lr

model.fit(train_batches,
          epochs = 5,
          validation_data = validation_batches,
          callbacks=[LearningRateScheduler(step_decay_with_epochs_drop, verbose=1),
                     TensorBoard(log_dir='./logs')])


Epoch 1: LearningRateScheduler setting learning rate to 0.005.
Epoch 1/5
26/26 ━━━━━━━━━━━━━━━━━━━━ 1s 39ms/step - accuracy: 0.6560 - loss: 0.6487 - val_accuracy: 0.7659 - val_loss: 0.6257 - learning_rate: 0.0050

Epoch 2: LearningRateScheduler setting learning rate to 0.0025.
Epoch 2/5
26/26 ━━━━━━━━━━━━━━━━━━━━ 1s 36ms/step - accuracy: 0.7405 - loss: 0.6186 - val_accuracy: 0.7171 - val_loss: 0.6198 - learning_rate: 0.0025

Epoch 3: LearningRateScheduler setting learning rate to 0.00125.
Epoch 3/5
26/26 ━━━━━━━━━━━━━━━━━━━━ 1s 38ms/step - accuracy: 0.7408 - loss: 0.6087 - val_accuracy: 0.6049 - val_loss: 0.6301 - learning_rate: 0.0012

Epoch 4: LearningRateScheduler setting learning rate to 0.000625.
Epoch 4/5
26/26 ━━━━━━━━━━━━━━━━━━━━ 1s 37ms/step - accuracy: 0.7229 - loss: 0.5972 - val_accuracy: 0.7171 - val_loss: 0.6071 - learning_rate: 6.2500e-04

Epoch 5: LearningRateScheduler setting learning rate to 0.0003125.
Epoch 5/5
26/26 ━━━━━━━━━━━━━━━━━━━━ 1s 37ms/step - accuracy: 0.75

In [29]:
def step_decay_without_epochs_drop(epoch):
    initial_lr = 0.01
    drop = 0.5
    lr = initial_lr * math.pow(drop, math.floor((1+epoch)))
    return lr

model.fit(train_batches,
          epochs = 5,
          validation_data = validation_batches,
          callbacks=[LearningRateScheduler(step_decay_without_epochs_drop, verbose=1),
                     TensorBoard(log_dir='./logs')])


Epoch 1: LearningRateScheduler setting learning rate to 0.005.
Epoch 1/5
26/26 ━━━━━━━━━━━━━━━━━━━━ 1s 40ms/step - accuracy: 0.7411 - loss: 0.5866 - val_accuracy: 0.7268 - val_loss: 0.5600 - learning_rate: 0.0050

Epoch 2: LearningRateScheduler setting learning rate to 0.0025.
Epoch 2/5
26/26 ━━━━━━━━━━━━━━━━━━━━ 1s 35ms/step - accuracy: 0.7546 - loss: 0.5475 - val_accuracy: 0.7317 - val_loss: 0.5704 - learning_rate: 0.0025

Epoch 3: LearningRateScheduler setting learning rate to 0.00125.
Epoch 3/5
26/26 ━━━━━━━━━━━━━━━━━━━━ 1s 36ms/step - accuracy: 0.7530 - loss: 0.5472 - val_accuracy: 0.7317 - val_loss: 0.5644 - learning_rate: 0.0012

Epoch 4: LearningRateScheduler setting learning rate to 0.000625.
Epoch 4/5
26/26 ━━━━━━━━━━━━━━━━━━━━ 1s 37ms/step - accuracy: 0.7737 - loss: 0.5143 - val_accuracy: 0.8000 - val_loss: 0.5313 - learning_rate: 6.2500e-04

Epoch 5: LearningRateScheduler setting learning rate to 0.0003125.
Epoch 5/5
26/26 ━━━━━━━━━━━━━━━━━━━━ 1s 34ms/step - accuracy: 0.80

In [32]:
%tensorboard --logdir logs

Reusing TensorBoard on port 6006 (pid 45920), started 7:34:18 ago. (Use '!kill 45920' to kill it.)